# EA-TTM — Slim Decoder Cross-Feed Audit and Cache

**Project:** Extreme-Aware Tiny Time Mixers (EA-TTM) for Retail Demand Forecasting under Regime Shifts  
**Phase:** 05 — Slim Decoder cross-feed contract  
**Scope:** Define a mathematically explicit, horizon-aligned decoder representation for the supervisor-approved EA-TTM architecture and cache the frozen HP-TTM outputs required by later adapter training.

This phase performs **no parameter training** and does **not** evaluate the locked final-test interval.

## Locked upstream interface

Phase 01 established the frozen TTM interfaces:

$$
\mathbf{D}
\in
\mathbb{R}^{B\times1\times9\times128},
$$

where $\mathbf{D}$ is the Slim TTM Decoder hidden state, and

$$
\hat{\mathbf{Y}}_{\mathrm{HP}}
\in
\mathbb{R}^{B\times28\times1},
$$

where $\hat{\mathbf{Y}}_{\mathrm{HP}}$ is the head-probed TTM base forecast.

The Forecasting Head first flattens the decoder tensor:

$$
9\times128=1152,
$$

and its linear forecast block produces 48 native forecast values before the configured 28-step prediction filtering.

## Horizon-aligned decoder representation

The supervisor-approved EA-TTM architecture requires decoder information $\mathbf{d}_h$ for each forecast horizon $h$.

A new trainable horizon projector would add an architectural component that is not present in the approved diagram. Instead, this implementation uses the **already trained and subsequently frozen HP-TTM Forecasting Head** to align the Slim Decoder representation to each horizon.

Let:

$$
D_{b,p,k}
$$

denote decoder feature $k$ at decoder patch/token $p$, where:

- $b$ indexes the sample;
- $p=1,\ldots,9$;
- $k=1,\ldots,128$.

Let the frozen Forecasting Head weight for horizon $h$ be reshaped as:

$$
W^{(h)}
\in
\mathbb{R}^{9\times128}.
$$

The horizon-aligned decoder feature is defined component-wise as:

$$
d_{b,h,k}
=
\sum_{p=1}^{9}
W^{(h)}_{p,k}
D_{b,p,k}.
$$

Therefore:

$$
\mathbf{d}_{b,h}
\in
\mathbb{R}^{128}.
$$

This construction has an exact reconstruction property:

$$
\hat y^{\mathrm{head}}_{b,h}
=
\sum_{k=1}^{128}
d_{b,h,k}
+
b_h,
$$

where $b_h$ is the corresponding frozen forecasting-head bias.

The representation is therefore:

- horizon-specific;
- derived directly from the Slim TTM Decoder;
- aligned using the already-trained HP-TTM Forecasting Head;
- free of additional trainable parameters;
- auditable through numerical reconstruction.

## Cache strategy

The complete HP-TTM is frozen before EA-TTM adapter training. Repeatedly executing a frozen model during every adapter epoch would waste compute.

This phase therefore caches:

1. the raw Slim Decoder hidden state $\mathbf{D}$;
2. the 28-step HP-TTM base forecast.

The horizon-aligned representation $\mathbf{d}_h$ is **not** redundantly stored. It is reconstructed cheaply from the cached decoder tensor and the saved frozen head weights.

For 121,960 training windows, caching the raw decoder state requires substantially less storage than caching the full $28\times128$ horizon-aligned representation.

In [1]:
from __future__ import annotations

import hashlib
import importlib.util
import json
import os
import random
import subprocess
import sys
import time
from pathlib import Path
from typing import Any

import numpy as np
import pandas as pd
import torch

## Runtime and project preflight

In [2]:
from google.colab import drive

drive.mount("/content/drive")

PROJECT_ROOT = Path("/content/drive/MyDrive/EA_TTM_RETAIL")

HEAD_PROBED_MODEL_DIR = PROJECT_ROOT / "models" / "head_probed_ttm"
PROCESSED_M5_DIR = PROJECT_ROOT / "data" / "processed" / "m5"
METADATA_DIR = PROJECT_ROOT / "metadata"
RESULTS_TABLE_DIR = PROJECT_ROOT / "results" / "tables"

RUN_DIR = PROJECT_ROOT / "runs" / "05_decoder_crossfeed"
CACHE_DIR = RUN_DIR / "cache"

for directory in (
    RESULTS_TABLE_DIR,
    RUN_DIR,
    CACHE_DIR,
):
    directory.mkdir(parents=True, exist_ok=True)

# Phase 05 no longer depends on the Phase 01 contract file.
# Everything required here is recovered from the frozen HP-TTM,
# the Phase 02 split manifest, the Phase 04 run summary, and the
# validated processed M5 data.
required_artifacts = {
    "head_probed_model_config": HEAD_PROBED_MODEL_DIR / "config.json",
    "split_manifest": METADATA_DIR / "02_SPLIT_MANIFEST.json",
    "phase04_summary": METADATA_DIR / "04_RUN_SUMMARY.json",
    "processed_sales": PROCESSED_M5_DIR / "sales_train_evaluation.parquet",
    "processed_calendar": PROCESSED_M5_DIR / "calendar.parquet",
}

artifact_preflight = {
    name: {
        "path": str(path),
        "exists": path.exists(),
    }
    for name, path in required_artifacts.items()
}

missing_artifacts = [
    details["path"]
    for details in artifact_preflight.values()
    if not details["exists"]
]

print("PHASE 05 PREREQUISITE AUDIT")
print(
    json.dumps(
        artifact_preflight,
        indent=2,
    )
)

assert not missing_artifacts, (
    "Phase 05 prerequisites are incomplete. Missing: "
    + "; ".join(missing_artifacts)
)

Mounted at /content/drive
PHASE 05 PREREQUISITE AUDIT
{
  "head_probed_model_config": {
    "path": "/content/drive/MyDrive/EA_TTM_RETAIL/models/head_probed_ttm/config.json",
    "exists": true
  },
  "split_manifest": {
    "path": "/content/drive/MyDrive/EA_TTM_RETAIL/metadata/02_SPLIT_MANIFEST.json",
    "exists": true
  },
  "phase04_summary": {
    "path": "/content/drive/MyDrive/EA_TTM_RETAIL/metadata/04_RUN_SUMMARY.json",
    "exists": true
  },
  "processed_sales": {
    "path": "/content/drive/MyDrive/EA_TTM_RETAIL/data/processed/m5/sales_train_evaluation.parquet",
    "exists": true
  },
  "processed_calendar": {
    "path": "/content/drive/MyDrive/EA_TTM_RETAIL/data/processed/m5/calendar.parquet",
    "exists": true
  }
}


In [3]:
TSFM_DISTRIBUTION_VERSION = "0.3.9"
TSFM_GIT_REPOSITORY = "https://github.com/ibm-granite/granite-tsfm.git"
TSFM_GIT_TAG = f"v{TSFM_DISTRIBUTION_VERSION}"
TSFM_SOURCE_LOCK = METADATA_DIR / "00_TSFM_SOURCE_LOCK.txt"

RUNTIME_VENDOR_ROOT = Path("/content/ea_ttm_vendor")
TSFM_SOURCE_ROOT = RUNTIME_VENDOR_ROOT / "granite-tsfm"
RUNTIME_VENDOR_ROOT.mkdir(parents=True, exist_ok=True)

def module_available(module_name: str) -> bool:
    return importlib.util.find_spec(module_name) is not None

def run_command(command: list[str], cwd: Path | None = None) -> str:
    result = subprocess.run(
        command,
        cwd=str(cwd) if cwd is not None else None,
        check=True,
        capture_output=True,
        text=True,
    )
    return result.stdout.strip()

if not module_available("tsfm_public"):
    subprocess.run(
        [
            sys.executable,
            "-m",
            "pip",
            "install",
            "-q",
            "--no-deps",
            f"granite-tsfm=={TSFM_DISTRIBUTION_VERSION}",
        ],
        check=True,
    )
    importlib.invalidate_caches()

if not module_available("tsfm_public"):
    if TSFM_SOURCE_LOCK.exists():
        tsfm_source_commit = TSFM_SOURCE_LOCK.read_text(
            encoding="utf-8"
        ).strip()
    else:
        tag_query = subprocess.run(
            [
                "git",
                "ls-remote",
                "--tags",
                TSFM_GIT_REPOSITORY,
                f"refs/tags/{TSFM_GIT_TAG}",
            ],
            check=False,
            capture_output=True,
            text=True,
        ).stdout.strip()

        if tag_query:
            tsfm_source_commit = tag_query.split()[0]
        else:
            tsfm_source_commit = run_command(
                [
                    "git",
                    "ls-remote",
                    TSFM_GIT_REPOSITORY,
                    "refs/heads/main",
                ]
            ).split()[0]

        TSFM_SOURCE_LOCK.write_text(
            tsfm_source_commit,
            encoding="utf-8",
        )

    if TSFM_SOURCE_ROOT.exists():
        subprocess.run(
            ["rm", "-rf", str(TSFM_SOURCE_ROOT)],
            check=True,
        )

    run_command(
        [
            "git",
            "clone",
            "--quiet",
            "--no-checkout",
            TSFM_GIT_REPOSITORY,
            str(TSFM_SOURCE_ROOT),
        ]
    )
    run_command(
        ["git", "checkout", "--quiet", tsfm_source_commit],
        cwd=TSFM_SOURCE_ROOT,
    )

    sys.path.insert(0, str(TSFM_SOURCE_ROOT))
    importlib.invalidate_caches()

assert module_available("tsfm_public")

from tsfm_public.models.tinytimemixer.modeling_tinytimemixer import (
    TinyTimeMixerForPrediction,
)
from tsfm_public.toolkit.time_series_preprocessor import (
    DEFAULT_FREQUENCY_MAPPING,
)

## Reproducibility state

The project seed remains:

$$
s=2026.
$$

In [4]:
BASE_SEED = 2026

def set_global_seed(seed: int) -> None:
    os.environ["PYTHONHASHSEED"] = str(seed)
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)

    torch.use_deterministic_algorithms(True, warn_only=True)

    if hasattr(torch.backends, "cudnn"):
        torch.backends.cudnn.benchmark = False
        torch.backends.cudnn.deterministic = True

set_global_seed(BASE_SEED)

assert torch.cuda.is_available(), "CUDA GPU is required for Phase 05."
DEVICE = torch.device("cuda")

## Upstream contract recovery and verification

Phase 05 intentionally avoids a hard dependency on the Phase 01 metadata file.

The cross-feed computation requires only:

- the frozen HP-TTM produced in Phase 04;
- the Phase 02 temporal split manifest;
- the Phase 04 run summary;
- the validated M5 sales and calendar artifacts.

The daily frequency is re-verified directly from the processed M5 calendar:

$$
t_{i+1}-t_i = 1\ \text{day},
\qquad
i=1,\ldots,1940.
$$

Only after this check is the IBM daily frequency token resolved.

In [5]:
split_manifest = json.loads(
    (METADATA_DIR / "02_SPLIT_MANIFEST.json").read_text(
        encoding="utf-8"
    )
)

phase04_summary = json.loads(
    (METADATA_DIR / "04_RUN_SUMMARY.json").read_text(
        encoding="utf-8"
    )
)

# Record the exact Phase 04 metadata schema before consuming it.
# This avoids assuming that fields printed in the Phase 04 RETURN BLOCK
# were also persisted in 04_RUN_SUMMARY.json.
phase04_available_keys = sorted(
    phase04_summary.keys()
)

required_phase04_keys = {
    "dataset_sha256",
    "split_sha256",
    "selected_training_batch_size",
    "head_probed_metrics",
}

missing_phase04_keys = sorted(
    required_phase04_keys
    - set(phase04_summary)
)

assert not missing_phase04_keys, (
    "Phase 04 run summary is missing required fields: "
    + ", ".join(missing_phase04_keys)
)

DATASET_SHA256 = phase04_summary["dataset_sha256"]
SPLIT_SHA256 = phase04_summary["split_sha256"]

CONTEXT_LENGTH = int(
    split_manifest["context_length"]
)

FORECAST_HORIZON = int(
    split_manifest["forecast_horizon"]
)

TRAIN_ORIGINS = [
    int(origin)
    for origin in split_manifest["train_origins"]
]

VALIDATION_ORIGIN = int(
    split_manifest["validation_origin"]
)

assert SPLIT_SHA256 == split_manifest["split_sha256"]

assert CONTEXT_LENGTH == 512
assert FORECAST_HORIZON == 28
assert TRAIN_ORIGINS == [512, 960, 1409, 1857]
assert VALIDATION_ORIGIN == 1885
assert split_manifest["locked_test_interval"] == [
    "d_1914",
    "d_1941",
]

# The earlier Phase 05 notebook incorrectly expected
# `selected_head_probed_model_frozen` inside 04_RUN_SUMMARY.json.
# That flag appeared in the Phase 04 RETURN BLOCK but was not persisted
# in the run-summary JSON. We therefore do not infer it from an absent key.
#
# The correct implementation-level verification is performed below:
# the HP-TTM snapshot must exist, load successfully, and every parameter
# is explicitly set to requires_grad=False before any Phase 05 forward pass.

calendar_frequency_audit = pd.read_parquet(
    PROCESSED_M5_DIR / "calendar.parquet",
    columns=["d", "date"],
)

calendar_frequency_audit = (
    calendar_frequency_audit
    .loc[
        calendar_frequency_audit["d"].isin(
            [
                f"d_{day}"
                for day in range(1, 1942)
            ]
        )
    ]
    .assign(
        day_index=lambda frame: (
            frame["d"]
            .str.removeprefix("d_")
            .astype("int32")
        )
    )
    .sort_values(
        "day_index",
        kind="stable",
    )
)

calendar_dates = pd.to_datetime(
    calendar_frequency_audit["date"]
)

day_differences = (
    calendar_dates
    .diff()
    .dropna()
    .dt.days
    .unique()
)

assert np.array_equal(
    np.sort(day_differences),
    np.asarray(
        [1],
        dtype=day_differences.dtype,
    ),
), (
    "The processed M5 calendar no longer represents "
    "a strictly daily observed-sales sequence."
)

FREQUENCY = "D"

FREQUENCY_TOKEN = int(
    DEFAULT_FREQUENCY_MAPPING[
        FREQUENCY
    ]
)

INFERENCE_BATCH_SIZE = int(
    phase04_summary[
        "selected_training_batch_size"
    ]
)

assert INFERENCE_BATCH_SIZE == 512

print(
    "Phase 04 run-summary keys:",
    phase04_available_keys,
)
print(
    "Recovered data frequency:",
    FREQUENCY,
)
print(
    "Resolved frequency token:",
    FREQUENCY_TOKEN,
)
print(
    "Cache inference batch size:",
    INFERENCE_BATCH_SIZE,
)

Phase 04 run-summary keys: ['backbone_trainable_parameters', 'best_epoch', 'dataset_sha256', 'decoder_trainable_parameters', 'early_stopping_patience', 'global_step', 'head_probe_vs_zero_shot', 'head_probed_metrics', 'head_trainable_parameters', 'max_epochs', 'selected_learning_rate', 'selected_training_batch_size', 'split_sha256', 'status', 'total_parameters', 'train_origins', 'trainable_parameters', 'training_config_sha256', 'training_runtime_seconds', 'validation_target', 'zero_shot_metrics']
Recovered data frequency: D
Resolved frequency token: 8
Cache inference batch size: 512


## Load and freeze HP-TTM

`save_pretrained()` does not persist Python `requires_grad` flags. The loaded HP-TTM is therefore explicitly frozen again before feature extraction.

In [6]:
hp_ttm = TinyTimeMixerForPrediction.from_pretrained(
    HEAD_PROBED_MODEL_DIR
).to(DEVICE)

# `save_pretrained()` serializes model weights/configuration, not Python
# requires_grad flags. Phase 05 therefore re-establishes and verifies the
# frozen contract from the actual loaded HP-TTM instance.
for parameter in hp_ttm.parameters():
    parameter.requires_grad_(False)

hp_ttm.eval()

loaded_hp_total_parameters = sum(
    parameter.numel()
    for parameter in hp_ttm.parameters()
)

loaded_hp_trainable_parameters = sum(
    parameter.numel()
    for parameter in hp_ttm.parameters()
    if parameter.requires_grad
)

assert loaded_hp_trainable_parameters == 0

assert hasattr(hp_ttm, "decoder")
assert hasattr(hp_ttm, "head")
assert hasattr(
    hp_ttm.head,
    "base_forecast_block",
)

forecast_linear = (
    hp_ttm.head.base_forecast_block
)

assert isinstance(
    forecast_linear,
    torch.nn.Linear,
), (
    "The audited Forecasting Head projection is "
    "no longer a Linear layer."
)

assert forecast_linear.in_features == 1152
assert forecast_linear.out_features == 48

print(
    "Loaded HP-TTM total parameters:",
    loaded_hp_total_parameters,
)
print(
    "Loaded HP-TTM trainable parameters:",
    loaded_hp_trainable_parameters,
)
print(
    "HP-TTM frozen contract: VERIFIED",
)

Loading weights:   0%|          | 0/139 [00:00<?, ?it/s]

Loaded HP-TTM total parameters: 799756
Loaded HP-TTM trainable parameters: 0
HP-TTM frozen contract: VERIFIED


## Construct development contexts

Only data through the validation target end (`d_1913`) is loaded. The locked final-test target is not materialized.

In [7]:
sales_path = PROCESSED_M5_DIR / "sales_train_evaluation.parquet"

development_columns = [
    f"d_{day}"
    for day in range(
        1,
        VALIDATION_ORIGIN + FORECAST_HORIZON + 1,
    )
]

development_frame = pd.read_parquet(
    sales_path,
    columns=development_columns,
)

development_values = development_frame.to_numpy(
    dtype=np.float32,
    copy=True,
)

series_count = development_values.shape[0]

training_contexts = np.empty(
    (
        series_count * len(TRAIN_ORIGINS),
        CONTEXT_LENGTH,
    ),
    dtype=np.float32,
)

for origin_index, origin in enumerate(TRAIN_ORIGINS):
    destination_start = origin_index * series_count
    destination_end = destination_start + series_count

    training_contexts[
        destination_start:destination_end
    ] = development_values[
        :,
        origin - CONTEXT_LENGTH:origin,
    ]

validation_context = development_values[
    :,
    VALIDATION_ORIGIN - CONTEXT_LENGTH:VALIDATION_ORIGIN,
]

validation_target = development_values[
    :,
    VALIDATION_ORIGIN:VALIDATION_ORIGIN + FORECAST_HORIZON,
]

assert training_contexts.shape == (121_960, 512)
assert validation_context.shape == (30_490, 512)
assert validation_target.shape == (30_490, 28)

## Empirical horizon-filter audit

The native Forecasting Head linear block produces 48 values:

$$
\mathbf{u}
\in
\mathbb{R}^{B\times1\times48}.
$$

The complete TTM head returns 28 values:

$$
\mathbf{v}
\in
\mathbb{R}^{B\times28\times1}.
$$

Rather than assuming which native horizons are retained, the notebook matches the observed tensors numerically and records the exact native horizon index corresponding to every returned horizon.

In [8]:
AUDIT_BATCH_SIZE = 64

audit_values = torch.from_numpy(
    validation_context[:AUDIT_BATCH_SIZE]
).unsqueeze(-1).to(DEVICE)

audit_frequency_token = torch.full(
    (AUDIT_BATCH_SIZE,),
    FREQUENCY_TOKEN,
    dtype=torch.long,
    device=DEVICE,
)

captured_internal_outputs: dict[str, torch.Tensor] = {}

def capture_native_forecast(
    module,
    inputs,
    output,
) -> None:
    captured_internal_outputs["native_forecast"] = (
        output.detach()
    )

def capture_head_output(
    module,
    inputs,
    output,
) -> None:
    captured_internal_outputs["head_output"] = (
        output.detach()
    )

native_handle = (
    hp_ttm.head.base_forecast_block
    .register_forward_hook(
        capture_native_forecast
    )
)

head_handle = (
    hp_ttm.head
    .register_forward_hook(
        capture_head_output
    )
)

try:
    with torch.no_grad():
        audit_output = hp_ttm(
            past_values=audit_values,
            freq_token=audit_frequency_token,
            output_hidden_states=True,
            return_dict=True,
        )
finally:
    native_handle.remove()
    head_handle.remove()

decoder_hidden_state = (
    audit_output.decoder_hidden_state
)

assert decoder_hidden_state.shape == (
    AUDIT_BATCH_SIZE,
    1,
    9,
    128,
)

native_forecast = captured_internal_outputs[
    "native_forecast"
]

filtered_head_output = captured_internal_outputs[
    "head_output"
]

assert native_forecast.shape == (
    AUDIT_BATCH_SIZE,
    1,
    48,
)

assert filtered_head_output.shape == (
    AUDIT_BATCH_SIZE,
    28,
    1,
)

native_forecast_bhc = native_forecast.permute(
    0,
    2,
    1,
)

matched_native_indices: list[int] = []
match_max_abs_errors: list[float] = []

for returned_horizon_index in range(
    FORECAST_HORIZON
):
    returned_values = filtered_head_output[
        :,
        returned_horizon_index,
        :,
    ]

    candidate_errors = torch.amax(
        torch.abs(
            native_forecast_bhc
            - returned_values.unsqueeze(1)
        ),
        dim=(0, 2),
    )

    matched_index = int(
        torch.argmin(candidate_errors).item()
    )

    matched_error = float(
        candidate_errors[
            matched_index
        ].item()
    )

    matched_native_indices.append(
        matched_index
    )

    match_max_abs_errors.append(
        matched_error
    )

assert len(set(matched_native_indices)) == FORECAST_HORIZON

assert max(match_max_abs_errors) < 1e-5, (
    "The 28 returned horizons could not be matched exactly "
    "to the 48 native Forecasting Head outputs."
)

print(
    "Returned-to-native horizon mapping:",
    matched_native_indices,
)
print(
    "Maximum horizon-mapping error:",
    max(match_max_abs_errors),
)

Returned-to-native horizon mapping: [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27]
Maximum horizon-mapping error: 0.0


## Exact decoder contribution reconstruction

The frozen linear Forecasting Head has weights:

$$
\mathbf{W}
\in
\mathbb{R}^{48\times1152}.
$$

For the 28 returned horizons, the selected weights are reshaped as:

$$
\mathbf{W}_{sel}
\in
\mathbb{R}^{28\times9\times128}.
$$

The horizon-aligned decoder cross-feed tensor is:

$$
\mathbf{D}^{(h)}
\in
\mathbb{R}^{B\times28\times128}.
$$

The numerical audit requires this representation to reconstruct the filtered Forecasting Head output to floating-point tolerance.

In [9]:
native_weight = (
    forecast_linear.weight
    .detach()
    .float()
)

native_bias = (
    forecast_linear.bias
    .detach()
    .float()
)

selected_weight = native_weight[
    matched_native_indices
].reshape(
    FORECAST_HORIZON,
    9,
    128,
)

selected_bias = native_bias[
    matched_native_indices
]

audit_decoder = (
    decoder_hidden_state[:, 0, :, :]
    .detach()
    .float()
)

decoder_horizon_features = torch.einsum(
    "bpk,hpk->bhk",
    audit_decoder,
    selected_weight,
)

reconstructed_head_output = (
    decoder_horizon_features.sum(dim=-1)
    + selected_bias.unsqueeze(0)
)

observed_head_output = (
    filtered_head_output
    .squeeze(-1)
    .float()
)

reconstruction_error = torch.abs(
    reconstructed_head_output
    - observed_head_output
)

reconstruction_max_abs_error = float(
    reconstruction_error.max().item()
)

reconstruction_mean_abs_error = float(
    reconstruction_error.mean().item()
)

assert decoder_horizon_features.shape == (
    AUDIT_BATCH_SIZE,
    28,
    128,
)

assert reconstruction_max_abs_error < 1e-4, (
    "Horizon-aligned decoder representation did not "
    "reconstruct the frozen Forecasting Head output."
)

print(
    "Decoder horizon feature shape:",
    tuple(decoder_horizon_features.shape),
)
print(
    "Head reconstruction max abs error:",
    reconstruction_max_abs_error,
)
print(
    "Head reconstruction mean abs error:",
    reconstruction_mean_abs_error,
)

Decoder horizon feature shape: (64, 28, 128)
Head reconstruction max abs error: 1.1920928955078125e-07
Head reconstruction mean abs error: 1.0395784677541542e-08


## Persist decoder-alignment parameters

The fixed head-derived alignment parameters are saved independently from the model so later EA-TTM notebooks can reconstruct the exact same decoder cross-feed tensor without model introspection.

In [10]:
ALIGNMENT_WEIGHTS_PATH = (
    METADATA_DIR
    / "05_DECODER_ALIGNMENT_WEIGHTS.npz"
)

np.savez_compressed(
    ALIGNMENT_WEIGHTS_PATH,
    native_horizon_indices=np.asarray(
        matched_native_indices,
        dtype=np.int32,
    ),
    selected_weight=(
        selected_weight
        .cpu()
        .numpy()
        .astype(np.float32)
    ),
    selected_bias=(
        selected_bias
        .cpu()
        .numpy()
        .astype(np.float32)
    ),
)

assert ALIGNMENT_WEIGHTS_PATH.exists()

## Resumable frozen-feature cache

Raw decoder states and HP-TTM forecasts are cached in deterministic sample order:

- training: origin-major, then store-item series order;
- validation: store-item series order.

Cache progress is persisted after every completed batch.

In [11]:
TRAIN_DECODER_CACHE_PATH = (
    CACHE_DIR
    / "train_decoder_hidden.npy"
)

TRAIN_FORECAST_CACHE_PATH = (
    CACHE_DIR
    / "train_base_forecast.npy"
)

VALIDATION_DECODER_CACHE_PATH = (
    CACHE_DIR
    / "validation_decoder_hidden.npy"
)

VALIDATION_FORECAST_CACHE_PATH = (
    CACHE_DIR
    / "validation_base_forecast.npy"
)

CACHE_PROGRESS_PATH = (
    METADATA_DIR
    / "05_CACHE_PROGRESS.json"
)

expected_cache_contract = {
    "dataset_sha256": DATASET_SHA256,
    "split_sha256": SPLIT_SHA256,
    "training_samples": int(
        training_contexts.shape[0]
    ),
    "validation_samples": int(
        validation_context.shape[0]
    ),
    "decoder_shape_per_sample": [
        1,
        9,
        128,
    ],
    "forecast_horizon": FORECAST_HORIZON,
    "dtype": "float32",
    "batch_size": INFERENCE_BATCH_SIZE,
}

cache_contract_sha256 = hashlib.sha256(
    json.dumps(
        expected_cache_contract,
        sort_keys=True,
        separators=(",", ":"),
    ).encode("utf-8")
).hexdigest()

if CACHE_PROGRESS_PATH.exists():
    cache_progress = json.loads(
        CACHE_PROGRESS_PATH.read_text(
            encoding="utf-8"
        )
    )

    assert (
        cache_progress["cache_contract_sha256"]
        == cache_contract_sha256
    )
else:
    cache_progress = {
        "cache_contract_sha256": cache_contract_sha256,
        "train_next_index": 0,
        "validation_next_index": 0,
        "train_complete": False,
        "validation_complete": False,
    }

def write_cache_progress() -> None:
    temporary_path = CACHE_PROGRESS_PATH.with_suffix(
        ".tmp"
    )

    temporary_path.write_text(
        json.dumps(
            cache_progress,
            indent=2,
        ),
        encoding="utf-8",
    )

    temporary_path.replace(
        CACHE_PROGRESS_PATH
    )

write_cache_progress()

In [12]:
def open_or_create_memmap(
    path: Path,
    shape: tuple[int, ...],
) -> np.memmap:
    if path.exists():
        array = np.lib.format.open_memmap(
            path,
            mode="r+",
        )

        assert array.shape == shape
        assert array.dtype == np.float32
        return array

    return np.lib.format.open_memmap(
        path,
        mode="w+",
        dtype=np.float32,
        shape=shape,
    )

train_decoder_cache = open_or_create_memmap(
    TRAIN_DECODER_CACHE_PATH,
    (
        training_contexts.shape[0],
        1,
        9,
        128,
    ),
)

train_forecast_cache = open_or_create_memmap(
    TRAIN_FORECAST_CACHE_PATH,
    (
        training_contexts.shape[0],
        FORECAST_HORIZON,
    ),
)

validation_decoder_cache = open_or_create_memmap(
    VALIDATION_DECODER_CACHE_PATH,
    (
        validation_context.shape[0],
        1,
        9,
        128,
    ),
)

validation_forecast_cache = open_or_create_memmap(
    VALIDATION_FORECAST_CACHE_PATH,
    (
        validation_context.shape[0],
        FORECAST_HORIZON,
    ),
)

## Frozen HP-TTM cache extraction

In [13]:
def cache_model_outputs(
    *,
    contexts: np.ndarray,
    decoder_cache: np.memmap,
    forecast_cache: np.memmap,
    progress_key: str,
    complete_key: str,
) -> None:
    if cache_progress[complete_key]:
        return

    start_index = int(
        cache_progress[progress_key]
    )

    hp_ttm.eval()

    with torch.no_grad():
        for batch_start in range(
            start_index,
            contexts.shape[0],
            INFERENCE_BATCH_SIZE,
        ):
            batch_end = min(
                batch_start + INFERENCE_BATCH_SIZE,
                contexts.shape[0],
            )

            batch_values = torch.from_numpy(
                contexts[batch_start:batch_end]
            ).unsqueeze(-1).to(DEVICE)

            batch_token = torch.full(
                (batch_end - batch_start,),
                FREQUENCY_TOKEN,
                dtype=torch.long,
                device=DEVICE,
            )

            output = hp_ttm(
                past_values=batch_values,
                freq_token=batch_token,
                output_hidden_states=True,
                return_dict=True,
            )

            decoder_hidden = (
                output.decoder_hidden_state
                .detach()
                .float()
                .cpu()
                .numpy()
            )

            base_forecast = (
                output.prediction_outputs
                .detach()
                .squeeze(-1)
                .float()
                .cpu()
                .numpy()
            )

            assert decoder_hidden.shape == (
                batch_end - batch_start,
                1,
                9,
                128,
            )

            assert base_forecast.shape == (
                batch_end - batch_start,
                FORECAST_HORIZON,
            )

            decoder_cache[
                batch_start:batch_end
            ] = decoder_hidden

            forecast_cache[
                batch_start:batch_end
            ] = base_forecast

            decoder_cache.flush()
            forecast_cache.flush()

            cache_progress[
                progress_key
            ] = batch_end

            write_cache_progress()

            del (
                batch_values,
                batch_token,
                output,
                decoder_hidden,
                base_forecast,
            )

    cache_progress[complete_key] = True
    write_cache_progress()

cache_start_time = time.perf_counter()

cache_model_outputs(
    contexts=training_contexts,
    decoder_cache=train_decoder_cache,
    forecast_cache=train_forecast_cache,
    progress_key="train_next_index",
    complete_key="train_complete",
)

cache_model_outputs(
    contexts=validation_context,
    decoder_cache=validation_decoder_cache,
    forecast_cache=validation_forecast_cache,
    progress_key="validation_next_index",
    complete_key="validation_complete",
)

cache_runtime_seconds = (
    time.perf_counter()
    - cache_start_time
)

assert cache_progress["train_complete"]
assert cache_progress["validation_complete"]

print(
    "Frozen-feature cache runtime (s):",
    round(cache_runtime_seconds, 3),
)

Frozen-feature cache runtime (s): 22.893


## Cache integrity and HP-TTM reproduction audit

The cached validation forecast must reproduce the Phase 04 HP-TTM validation result within a small numerical tolerance.

In [14]:
def forecasting_metrics(
    actual: np.ndarray,
    predicted: np.ndarray,
) -> dict[str, float]:
    actual_64 = actual.astype(
        np.float64,
        copy=False,
    )

    predicted_64 = predicted.astype(
        np.float64,
        copy=False,
    )

    error = predicted_64 - actual_64
    absolute_error = np.abs(error)
    squared_error = np.square(error)

    mse = float(squared_error.mean())
    rmse = float(np.sqrt(mse))
    mae = float(absolute_error.mean())

    denominator = float(
        np.abs(actual_64).sum()
    )

    wape = float(
        absolute_error.sum()
        / denominator
    )

    return {
        "mse": mse,
        "rmse": rmse,
        "mae": mae,
        "wape": wape,
    }

cached_validation_forecast = np.asarray(
    validation_forecast_cache
)

cached_validation_metrics = forecasting_metrics(
    validation_target,
    cached_validation_forecast,
)

phase04_metrics = phase04_summary[
    "head_probed_metrics"
]

for metric_name in (
    "mse",
    "rmse",
    "mae",
    "wape",
):
    assert np.isclose(
        cached_validation_metrics[metric_name],
        phase04_metrics[metric_name],
        rtol=5e-3,
        atol=5e-3,
    ), (
        f"Cached HP-TTM {metric_name} does not reproduce "
        "the Phase 04 validation result."
    )

assert np.isfinite(
    np.asarray(train_decoder_cache)
).all()

assert np.isfinite(
    np.asarray(train_forecast_cache)
).all()

assert np.isfinite(
    np.asarray(validation_decoder_cache)
).all()

assert np.isfinite(
    np.asarray(validation_forecast_cache)
).all()

## Alignment diagnostic table

In [15]:
selected_weight_cpu = (
    selected_weight
    .detach()
    .cpu()
    .numpy()
)

alignment_records: list[dict[str, Any]] = []

for returned_horizon_index, native_index in enumerate(
    matched_native_indices
):
    horizon_weight = selected_weight_cpu[
        returned_horizon_index
    ]

    alignment_records.append(
        {
            "returned_horizon": (
                returned_horizon_index + 1
            ),
            "native_horizon_index_zero_based": (
                native_index
            ),
            "native_horizon_one_based": (
                native_index + 1
            ),
            "mapping_max_abs_error": (
                match_max_abs_errors[
                    returned_horizon_index
                ]
            ),
            "weight_l2_norm": float(
                np.linalg.norm(
                    horizon_weight
                )
            ),
            "bias": float(
                selected_bias[
                    returned_horizon_index
                ].item()
            ),
        }
    )

alignment_table = pd.DataFrame(
    alignment_records
)

alignment_table

,returned_horizon,native_horizon_index_zero_based,native_horizon_one_based,mapping_max_abs_error,weight_l2_norm,bias
0,1,0,1,0.0,1.019346,-0.005995
1,2,1,2,0.0,0.878066,-0.020216
2,3,2,3,0.0,0.835574,-0.023713
3,4,3,4,0.0,0.856997,-0.025702
4,5,4,5,0.0,0.848644,-0.022000
5,6,5,6,0.0,0.884424,-0.016547
6,7,6,7,0.0,0.889974,-0.011039
7,8,7,8,0.0,0.863690,-0.013266
8,9,8,9,0.0,0.833387,-0.018181
9,10,9,10,0.0,0.846269,-0.012376


## Persist cross-feed contract

In [16]:
ALIGNMENT_TABLE_PATH = (
    RESULTS_TABLE_DIR
    / "05_DECODER_HORIZON_ALIGNMENT.csv"
)

CROSSFEED_CONTRACT_PATH = (
    METADATA_DIR
    / "05_DECODER_CROSSFEED_CONTRACT.json"
)

CACHE_MANIFEST_PATH = (
    METADATA_DIR
    / "05_CACHE_MANIFEST.json"
)

RUN_SUMMARY_PATH = (
    METADATA_DIR
    / "05_RUN_SUMMARY.json"
)

alignment_table.to_csv(
    ALIGNMENT_TABLE_PATH,
    index=False,
)

crossfeed_contract = {
    "status": "DECODER_CROSSFEED_CONTRACT_LOCKED",
    "source_model": "frozen_head_probed_ttm",
    "raw_decoder_shape": [
        "B",
        1,
        9,
        128,
    ],
    "raw_decoder_feature_dimension": 128,
    "decoder_patch_count": 9,
    "native_forecast_length": 48,
    "returned_forecast_length": 28,
    "matched_native_indices_zero_based": matched_native_indices,
    "horizon_aligned_feature_shape": [
        "B",
        28,
        128,
    ],
    "construction": (
        "d[b,h,k] = sum_p W_selected[h,p,k] * D[b,p,k]"
    ),
    "additional_trainable_parameters": 0,
    "loaded_hp_ttm_total_parameters": int(
        loaded_hp_total_parameters
    ),
    "loaded_hp_ttm_trainable_parameters": int(
        loaded_hp_trainable_parameters
    ),
    "reconstruction_max_abs_error": (
        reconstruction_max_abs_error
    ),
    "reconstruction_mean_abs_error": (
        reconstruction_mean_abs_error
    ),
    "alignment_weights_path": str(
        ALIGNMENT_WEIGHTS_PATH
    ),
}

crossfeed_payload = json.dumps(
    crossfeed_contract,
    sort_keys=True,
    separators=(",", ":"),
).encode("utf-8")

crossfeed_contract_sha256 = hashlib.sha256(
    crossfeed_payload
).hexdigest()

crossfeed_contract[
    "contract_sha256"
] = crossfeed_contract_sha256

CROSSFEED_CONTRACT_PATH.write_text(
    json.dumps(
        crossfeed_contract,
        indent=2,
    ),
    encoding="utf-8",
)

cache_manifest = {
    "cache_contract_sha256": (
        cache_contract_sha256
    ),
    "dataset_sha256": DATASET_SHA256,
    "split_sha256": SPLIT_SHA256,
    "sample_order": {
        "train": (
            "origin-major then M5 store-item series order"
        ),
        "validation": (
            "M5 store-item series order"
        ),
    },
    "train_decoder_hidden": {
        "path": str(
            TRAIN_DECODER_CACHE_PATH
        ),
        "shape": [
            121_960,
            1,
            9,
            128,
        ],
        "dtype": "float32",
    },
    "train_base_forecast": {
        "path": str(
            TRAIN_FORECAST_CACHE_PATH
        ),
        "shape": [
            121_960,
            28,
        ],
        "dtype": "float32",
    },
    "validation_decoder_hidden": {
        "path": str(
            VALIDATION_DECODER_CACHE_PATH
        ),
        "shape": [
            30_490,
            1,
            9,
            128,
        ],
        "dtype": "float32",
    },
    "validation_base_forecast": {
        "path": str(
            VALIDATION_FORECAST_CACHE_PATH
        ),
        "shape": [
            30_490,
            28,
        ],
        "dtype": "float32",
    },
}

CACHE_MANIFEST_PATH.write_text(
    json.dumps(
        cache_manifest,
        indent=2,
    ),
    encoding="utf-8",
)

run_summary = {
    "status": "PHASE_05_COMPLETE",
    "dataset_sha256": DATASET_SHA256,
    "split_sha256": SPLIT_SHA256,
    "crossfeed_contract_sha256": (
        crossfeed_contract_sha256
    ),
    "cache_contract_sha256": (
        cache_contract_sha256
    ),
    "raw_decoder_shape": [
        "B",
        1,
        9,
        128,
    ],
    "horizon_aligned_decoder_shape": [
        "B",
        28,
        128,
    ],
    "matched_native_indices_zero_based": (
        matched_native_indices
    ),
    "additional_trainable_parameters": 0,
    "loaded_hp_ttm_total_parameters": int(
        loaded_hp_total_parameters
    ),
    "loaded_hp_ttm_trainable_parameters": int(
        loaded_hp_trainable_parameters
    ),
    "reconstruction_max_abs_error": (
        reconstruction_max_abs_error
    ),
    "reconstruction_mean_abs_error": (
        reconstruction_mean_abs_error
    ),
    "cache_runtime_seconds": float(
        cache_runtime_seconds
    ),
    "cached_validation_metrics": (
        cached_validation_metrics
    ),
}

RUN_SUMMARY_PATH.write_text(
    json.dumps(
        run_summary,
        indent=2,
    ),
    encoding="utf-8",
)

1237

In [17]:
status_path = PROJECT_ROOT / "PROJECT_STATUS.md"

status_lines = [
    "# EA-TTM Project Status",
    "",
    "## Current stage",
    "Phase 05 — Slim Decoder cross-feed contract completed.",
    "",
    "## Decoder cross-feed",
    "- Raw HP-TTM decoder state: [B, 1, 9, 128]",
    "- Horizon-aligned decoder state: [B, 28, 128]",
    "- Additional trainable alignment parameters: 0",
    (
        "- Head reconstruction max absolute error: "
        f"{reconstruction_max_abs_error:.8g}"
    ),
    f"- Contract SHA-256: `{crossfeed_contract_sha256}`",
    "",
    "## Frozen HP-TTM cache",
    "- Training decoder states: 121,960 samples",
    "- Validation decoder states: 30,490 samples",
    "- Training base forecasts: 121,960 samples",
    "- Validation base forecasts: 30,490 samples",
    f"- Cache contract SHA-256: `{cache_contract_sha256}`",
    "",
    "## Next phase",
    (
        "Construct leakage-safe recent-demand signals and "
        "future-known retail covariates in the exact cache sample order."
    ),
]

status_path.write_text(
    "\n".join(status_lines),
    encoding="utf-8",
)

817

## Phase validation

Phase 05 passes only if:

$$
\begin{aligned}
&\mathbf{D}
\in
\mathbb{R}^{B\times1\times9\times128}
\\
&\land\
\mathbf{d}
\in
\mathbb{R}^{B\times28\times128}
\\
&\land\
\text{head reconstruction error}<10^{-4}
\\
&\land\
\text{additional alignment parameters}=0
\\
&\land\
\text{training and validation caches are complete}
\\
&\land\
\text{cached validation forecast reproduces HP-TTM}.
\end{aligned}
$$

In [18]:
phase_validation = {
    "decoder_interface_valid": (
        tuple(decoder_hidden_state.shape[1:])
        == (1, 9, 128)
    ),
    "horizon_decoder_shape_valid": (
        tuple(
            decoder_horizon_features.shape[1:]
        )
        == (28, 128)
    ),
    "exact_head_reconstruction": (
        reconstruction_max_abs_error
        < 1e-4
    ),
    "zero_added_alignment_parameters": True,
    "loaded_hp_ttm_frozen": (
        loaded_hp_trainable_parameters == 0
    ),
    "train_cache_complete": bool(
        cache_progress["train_complete"]
    ),
    "validation_cache_complete": bool(
        cache_progress["validation_complete"]
    ),
    "crossfeed_contract_saved": (
        CROSSFEED_CONTRACT_PATH.exists()
    ),
    "cache_manifest_saved": (
        CACHE_MANIFEST_PATH.exists()
    ),
    "run_summary_saved": (
        RUN_SUMMARY_PATH.exists()
    ),
}

assert all(
    phase_validation.values()
), (
    f"Phase 05 validation failed: "
    f"{phase_validation}"
)

phase_validation

{'decoder_interface_valid': True,
 'horizon_decoder_shape_valid': True,
 'exact_head_reconstruction': True,
 'zero_added_alignment_parameters': True,
 'loaded_hp_ttm_frozen': True,
 'train_cache_complete': True,
 'validation_cache_complete': True,
 'crossfeed_contract_saved': True,
 'cache_manifest_saved': True,
 'run_summary_saved': True}

## Return block

The final block is the **only successful-run output that needs to be returned to ChatGPT**.

In [19]:
return_block = {
    "status": "PHASE_05_COMPLETE",
    "dataset_sha256": DATASET_SHA256,
    "split_sha256": SPLIT_SHA256,
    "crossfeed_contract_sha256": (
        crossfeed_contract_sha256
    ),
    "cache_contract_sha256": (
        cache_contract_sha256
    ),
    "raw_decoder_shape": [
        "B",
        1,
        9,
        128,
    ],
    "horizon_aligned_decoder_shape": [
        "B",
        28,
        128,
    ],
    "matched_native_indices_zero_based": (
        matched_native_indices
    ),
    "matched_native_indices_one_based": [
        index + 1
        for index in matched_native_indices
    ],
    "additional_trainable_alignment_parameters": 0,
    "loaded_hp_ttm_total_parameters": int(
        loaded_hp_total_parameters
    ),
    "loaded_hp_ttm_trainable_parameters": int(
        loaded_hp_trainable_parameters
    ),
    "reconstruction_max_abs_error": (
        reconstruction_max_abs_error
    ),
    "reconstruction_mean_abs_error": (
        reconstruction_mean_abs_error
    ),
    "cache_runtime_seconds": float(
        cache_runtime_seconds
    ),
    "cache_counts": {
        "training": int(
            training_contexts.shape[0]
        ),
        "validation": int(
            validation_context.shape[0]
        ),
    },
    "cached_validation_metrics": (
        cached_validation_metrics
    ),
    "phase04_validation_metrics": (
        phase04_metrics
    ),
    "artifacts": {
        "decoder_crossfeed_contract": str(
            CROSSFEED_CONTRACT_PATH
        ),
        "alignment_weights": str(
            ALIGNMENT_WEIGHTS_PATH
        ),
        "alignment_table": str(
            ALIGNMENT_TABLE_PATH
        ),
        "cache_manifest": str(
            CACHE_MANIFEST_PATH
        ),
        "cache_progress": str(
            CACHE_PROGRESS_PATH
        ),
        "train_decoder_hidden": str(
            TRAIN_DECODER_CACHE_PATH
        ),
        "train_base_forecast": str(
            TRAIN_FORECAST_CACHE_PATH
        ),
        "validation_decoder_hidden": str(
            VALIDATION_DECODER_CACHE_PATH
        ),
        "validation_base_forecast": str(
            VALIDATION_FORECAST_CACHE_PATH
        ),
        "run_summary": str(
            RUN_SUMMARY_PATH
        ),
        "project_status": str(
            status_path
        ),
    },
}

print("PHASE 05 RETURN BLOCK")
print(
    json.dumps(
        return_block,
        indent=2,
    )
)

PHASE 05 RETURN BLOCK
{
  "status": "PHASE_05_COMPLETE",
  "dataset_sha256": "83962ce8bbcb0757215d045f81cac3a7cf4465943334ffa168af326bedc5ed59",
  "split_sha256": "5ff2e30f43a832bb9e513153bb15daa4e1e67c871b43586f2f2f11ef51396447",
  "crossfeed_contract_sha256": "064448a10cc1fe2b3e9744a94448f83865a3f8ddbec2e315bfcf09e94b93ffb2",
  "cache_contract_sha256": "ce1996174126f8533098e791765c02dfcdcb0e019f20cf96b4feb10d56c3b9b8",
  "raw_decoder_shape": [
    "B",
    1,
    9,
    128
  ],
  "horizon_aligned_decoder_shape": [
    "B",
    28,
    128
  ],
  "matched_native_indices_zero_based": [
    0,
    1,
    2,
    3,
    4,
    5,
    6,
    7,
    8,
    9,
    10,
    11,
    12,
    13,
    14,
    15,
    16,
    17,
    18,
    19,
    20,
    21,
    22,
    23,
    24,
    25,
    26,
    27
  ],
  "matched_native_indices_one_based": [
    1,
    2,
    3,
    4,
    5,
    6,
    7,
    8,
    9,
    10,
    11,
    12,
    13,
    14,
    15,
    16,
    17,
    18,
    19,
    2